# Visión por Computadora II #

## CEIA 21Co2025 ##

## TP Integrador ##

José Luis Diaz (diazjoseluis@gmail.com)

Ricardo Silvera (rsilvera@thalu.com.ar)

José Aviani (jose.aviani@gmail.com)


---

### Dataset: ###

#### 'X-Ray Baggage Scanner Anomaly Detection' en Kaggle ####

https://www.kaggle.com/datasets/orvile/x-ray-baggage-anomaly-detection/ 


---

Importar librerias:

In [ ]:
%pip install gdown
%pip install torch
%pip install torchvision

In [ ]:
import os
import zipfile
import gdown
import shutil

import glob

import torch
import torchvision

---

#### Descargar dataset ####

In [ ]:
data_dir = os.path.join(os.getcwd(), "data")

def descargar_dataset(data_dir):

  # Verificar si exsite la carpeta
  if not os.path.exists(data_dir):

    try:
      # Nombre del archivo ZIP que se va a guardar
      ZIP_NAME = "kaggle-xray_baggage_scanner_anomaly_detection.zip"
      zip_path = os.path.join(data_dir, ZIP_NAME)

      # Crear carpeta data
      os.makedirs(data_dir, exist_ok=True)

      # Descargar el ZIP
      print("Descargando archivo zip ...")
      # Lo tomamos de Google Drive porque Kaggle requeire autenticación
      gdown.download(id="1IqPblTm7nmKFpHXtl4beopE_SajTBoI0", output=zip_path, quiet=False)
      print("Archivo zip descargado.")

      # Descomprimir el ZIP
      print("Descomprimiendo archivo ...")
      with zipfile.ZipFile(zip_path, "r") as zf:
          zf.extractall(data_dir)
      print("Archivo descomprimido.")
    except:
      # En caso de error, eliminar la carpeta creada
      shutil.rmtree(data_dir, ignore_errors=True)
      print("Ocurrió un error al descargar el dataset.")

  print(f"Dataset descargado en: '{data_dir}'")


# Descargar el dataset (solo si no existe la carpeta data)
descargar_dataset(data_dir)

---

#### Cargar dataset ####

In [ ]:
# Dataset YOLO

CLASS_NAMES = ['gun', 'knife', 'pliers', 'scissors', 'wrench']

class YoloDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, dir_name, transforms=None):
        dir_path = os.path.join(data_dir, dir_name)
        images_dir = os.path.join(dir_path, "images")
        labels_dir = os.path.join(dir_path, "labels")
        
        self.images = sorted(glob.glob(os.path.join(images_dir, "*")))
        self.labels_dir = labels_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.images)

    def _read_yolo_txt(self, label_path):
        boxes_cxcywh = []
        labels = []
        if not os.path.exists(label_path):
            return torch.zeros((0,4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
        with open(label_path, "r") as f:
            lines = [ln.strip() for ln in f.readlines() if ln.strip()]
        if not lines:
            return torch.zeros((0,4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
        for ln in lines:
            c, cx, cy, w, h = ln.split()
            labels.append(int(c))
            boxes_cxcywh.append([float(cx), float(cy), float(w), float(h)])
        return torch.tensor(boxes_cxcywh, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(self.labels_dir, base + ".txt")
        boxes_cxcywh_norm, labels = self._read_yolo_txt(lbl_path)

        # Desnormalizar a píxeles
        if boxes_cxcywh_norm.numel() > 0:
            scale = torch.tensor([W, H, W, H], dtype=torch.float32)
            boxes_cxcywh_px = boxes_cxcywh_norm * scale
            boxes_xyxy = Image.box_convert(boxes_cxcywh_px, in_fmt="cxcywh", out_fmt="xyxy")
            # Clampear por seguridad
            boxes_xyxy[:, [0,2]] = boxes_xyxy[:, [0,2]].clamp(0, W)
            boxes_xyxy[:, [1,3]] = boxes_xyxy[:, [1,3]].clamp(0, H)
        else:
            boxes_xyxy = torch.zeros((0,4), dtype=torch.float32)

        target = {
            "boxes": boxes_xyxy,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": ((boxes_xyxy[:,2]-boxes_xyxy[:,0]) * (boxes_xyxy[:,3]-boxes_xyxy[:,1]))
              if boxes_xyxy.numel() else torch.tensor([], dtype=torch.float32),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)


In [ ]:
dataloader_batch_size = 32

transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

train_dataset = YoloDetectionDataset("train", transforms=transforms)
valid_dataset = YoloDetectionDataset("valid", transforms=transforms)
test_dataset = YoloDetectionDataset("test", transforms=transforms)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=dataloader_batch_size, shuffle=True, collate_fn=collate_fn)

In [ ]:
print (f"Imágenes de train: {len(train_dataloader.dataset)}")
print (f"Imágenes de valid: {len(valid_dataloader.dataset)}")
print (f"Imágenes de test: {len(test_dataloader.dataset)}")

---